# **TITAN Tutorial - Parameter Testing**

---

This notebook will test out two parameters: the minimum dBZ threshold and the minimum storm size. ***Users must run Titan_tutorial.ipynb first.***

---


### Note on task cells

This notebook uses two colored cells to indicate tasks.

<div class="alert alert-block alert-info"> <b>File Task: modify parameters in text files.</b> 

These text blocks help the user modify the parameter files or other functions in *external* text files.

</div>

<div class="alert alert-block alert-warning"> <b>Cell Task: run a command in Jupyter notebook cell.</b> 

These text blocks instruct the users to run a command *in* a cell within the Jupyter notebook. If you prefer, you are welcome to copy the commands (minus the ! symbol) into a terminal window.

</div>

---

## **1. Setup**
### Environment and packages

First, we import the required python packages to run this notebook. The LROSE processing can be done with the os package and shell commands.

In [ ]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

### 1.1 Set up directories

We need to set up the required data directories. 

In [ ]:
# make overall TITAN directories and ascii subdirectories for each parameter
!mkdir -p ./data/titan_min_dbz/ascii
!mkdir -p ./data/titan_size/ascii


### 1.2 Set up the environment

First, we'll set some key variables we'll need throughout the workflow.

In [ ]:
# Set directory variable to call LROSE
os.environ["LROSE_DIR"] = "/usr/local/lrose/bin"

## **2. Run TITAN storm tracking - minimum threshold**

In this section, we will focus on the *low_dbz_threshold* parameter. We have provided all other basic parameters including variable names, directories, etc.

A suggested range of parameter values is: . But feel free to try out different values to see how the analysis is affected.

<div class="alert alert-block alert-info"> <b>File Task: modify low_dbz_threshold parameter in text file.</b> 

To modify low_dbz_threshold, open the parameter file: ./params/Titan.params.mindbz, which you can edit directly in JupyterLab. Make sure to save the file.

</div>

Run TITAN on the same gridded data as in Titan_tutorial.ipynb to identify and track the storms.

NOTE: *TITAN requires a sounding to convert reflectivity data into meaningful storm metrics for the analysis. The sounding can be ingested in SPDB format or entered manually. In our case, we manually entered 10 levels retrieved from an ERA5 sounding corresponding to the radar's location and the date and time of this case. You can verify this by inspecting the parameter files and searching for "sounding_mode = SPECIFY_SOUNDING;", along with the manually entered array in that section.*

<div class="alert alert-block alert-warning"> <b>Task #1: Run Titan on the Jastrebac data with your chosen low_dbz_threshold value.</b>
<br>
    Note the different parameter file name and output directory.
    <br>
<code lang="bash">!$LROSE_DIR/Titan -params ./params/Titan.params.mindbz -start "2017 08 12 16 00 00" -end "2017 08 12 17 00 00" -odir ./data/titan_min_dbz/storms/ -debug</code>

</div>

In [ ]:
# run Titan
!$LROSE_DIR/Titan -params ./params/Titan.params.mindbz -start "2017 08 12 16 00 00" -end "2017 08 12 17 00 00" -odir ./data/titan_min_dbz/storms/ -debug  


### **2.2. Convert TITAN binary output to readable format - minimum threshold**

The TITAN output is in a binary format. In order to read the data, we first convert the TITAN output to an ASCII file.

<div class="alert alert-block alert-warning"> <b>Task #2: Convert Titan binary output to ASCII.</b>

<code lang="bash">!$LROSE_DIR/Tracks2Ascii -params ./params/Tracks2Ascii.params -f ./data/titan_min_dbz/storms/20170812.th5 > ./data/titan_min_dbz/ascii/Tracks2Ascii20170812.txt -debug</code>

</div>

In [ ]:
!$LROSE_DIR/Tracks2Ascii -params ./params/Tracks2Ascii.params -f ./data/titan_min_dbz/storms/20170812.th5 > ./data/titan_min_dbz/ascii/Tracks2Ascii20170812.txt -debug


### **2.3. Investigate Output - minimum threshold**

We'll load the necessary Python packages and plot some of the TITAN output now.

In [ ]:
# Import Python packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from shapely.geometry import Polygon
from netCDF4 import num2date
import netCDF4 as nc

In [ ]:
# set the path for the ASCII file
file_min_dbz = "./data/titan_min_dbz/ascii/Tracks2Ascii20170812.txt"

### **2.3.1 Read the ASCII file into a pandas dataframe**

Open text file and adjust columns names in order to import to a pandas dataframe. 
Since the text file has irregular delimiters, we need to add some extra steps.

In [ ]:
##open file and extract column names
f_min_dbz = open(file_min_dbz)
lines_min_dbz = f_min_dbz.readlines()
f_min_dbz.close()

label_line_index = None  

for i, line in enumerate(lines_min_dbz):
    if 'labels' in line:
        label_line_index = i
        break  
labels_min_dbz = lines_min_dbz[label_line_index].split(":", 1)[1].strip().split(",")

In [ ]:
#the data lines are the ones that do not start with #
data_lines_min_dbz = [line.strip() for line in lines_min_dbz if not line.startswith("#")]

The file last three rows are labeled "parents", "children", "nPolySidesPolygonRays*72". 

Parents and children columns refer to identifiers based on storm merging and splitting processes. The Polygon column shows the values for the lines from the polygon centroid to each vertex, in km. There are 72 values because each line is separated 5 deg (72*5 =360). 

With that information and the "envelope_centroid" column, we can retrieve the cells envelopes at each timestep.

In [ ]:
rows_min_dbz = []
for line in data_lines_min_dbz:
    parts = line.split()

    try:
        # Try parsing the polygon count value (always right before 72 values)
        poly_count_index = -73  # 72 floats + 1 count (the column starts with the numnber 72, which is not part of the values)

        # Parents and children may be missing
        parent_str = parts[poly_count_index - 2]
        child_str = parts[poly_count_index - 1]

        # Handle missing values marked as "-"
        parents = int(parent_str) if parent_str != '-' else np.nan
        children = int(child_str) if child_str != '-' else np.nan

        # Polygon values: skip the count, get the next 72 values
        polygon_values = list(map(float, parts[poly_count_index + 1:]))

        # Fixed columns
        fixed_cols = parts[:poly_count_index - 2]

        # Combine into one row
        row = fixed_cols + [parents, children, polygon_values]
        rows_min_dbz.append(row)
    except Exception as e:
        continue

In [ ]:
# Final columns: fixed + 3 custom ones
final_labels_min_dbz = labels_min_dbz[:len(rows_min_dbz[0]) - 3] + ['parents', 'children', 'nPolySidesPolygonRays']

# Create DataFrame
df_min_dbz = pd.DataFrame(rows_min_dbz, columns=final_labels_min_dbz)

# Convert date and time columns to datetime
df_min_dbz['date_utc'] = pd.to_datetime(
    df_min_dbz['Year'].astype(str) + '-' + df_min_dbz['Month'].astype(str).str.zfill(2) + '-' +
    df_min_dbz['Day'].astype(str).str.zfill(2) + ' ' + df_min_dbz['Hour'].astype(str).str.zfill(2) + ':' +
    df_min_dbz['Min'].astype(str).str.zfill(2) + ':' + df_min_dbz['Sec'].astype(str).str.zfill(2),
    format='%Y-%m-%d %H:%M:%S', utc=True
)
# Print df 
print(df_min_dbz)

Let's explore the TITAN output now!

### **2.3.2 Examine the number of identified individual storms per complex**

TITAN outputs cell features at each tracking timestep and identifies individual cells within larger systems based on their interaction with neighboring cells. As a result, each cell in the TITAN output ASCII file is assigned two identifiers: a "SimpleNum" and a "ComplexNum". For example, when tracking a multicell system, each individual cell within it will have a distinct "SimpleNum," but all cells belonging to that same system will share a common "ComplexNum."

Let's inspect our case now! How many Complexes can we identify? Which one contains more tracks (e.g., single cell tracks, and split/merge processes)?

In [ ]:
# Count number of unique SimpleNum per ComplexNum
simple_counts_min_dbz = df_min_dbz.groupby('ComplexNum')['SimpleNum'].nunique().reset_index(name='NumSimple')

# Sort (optional, for better visuals)
#simple_counts_min_dbz = simple_counts_min_dbz.sort_values(by='NumSimple', ascending=False)

# Plot 1
plt.figure(figsize=(10, 8))
sns.barplot(y='ComplexNum', x='NumSimple', data=simple_counts_min_dbz, palette='Set3')
plt.title('Number of SimpleNum per ComplexNum', fontsize=16)
plt.xlabel('Count of Unique SimpleNum ID', fontsize=14)
plt.ylabel('ComplexNum ID', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.grid(axis='x')
plt.show()


### **2.3.3 Explore a single complex**

Based on the figure above, let's now explore one of the Complex tracks. Look for a ComplexNum that has a lot of tracked individual storms. **Replace the number in the complexnum variable in the cell below.**

In our case, we will plot the Maximum Reflectivity (```'MaxDBZ(dBZ)'```) for the entire life cycle of this complex system. Each individual storm is plotted in a different color (see the ```hue='SimpleNum'``` parameter in the plot).

You can play and choose another attribute (e.g., Echo Top, Vil) from the ASCII file, and see how the attributes vary. **Copy the variable name from the ASCII file and replace the string in the plotvar variable in the cell below.**

In [1]:
# MODIFY THESE VARIABLES AS NEEDED

complexnum_min_dbz = "17" # should be a string!
plotvar_min_dbz = 'MaxDBZ(dBZ)'

In [ ]:
#  Filter dataframe for the chosen ComplexNum and sort by time
df0_min_dbz = df_min_dbz[dfdf_min_dbz['ComplexNum'] == complexnum_min_dbz].copy()
df0_min_dbz = df0_min_dbz.sort_values('date_utc')
df0_min_dbz[plotvar] = pd.to_numeric(df0_min_dbz[plotvar], errors='coerce')
df0_min_dbz['date_utc'] = pd.to_datetime(df0_min_dbz['date_utc'], errors='coerce', utc=True)

#  Plot
y_min = 30
y_max = 70
y_ticks = np.arange(y_min, y_max + 1, 5)

plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")
sns.lineplot(data=df0_min_dbz, x='date_utc', y=plotvar_min_dbz, hue='SimpleNum', palette='gist_ncar')
plt.ylim(y_min, y_max)
plt.yticks(y_ticks,fontsize=12)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.xticks(fontsize=12)
plt.title(plotvar_min_dbz+' over Time for ComplexNum = '+complexnum_min_dbz, fontsize=16)
plt.xlabel('Time (UTC)', fontsize=14)
plt.ylabel(plotvar_min_dbz,fontsize=14)
# Remove legend
plt.legend([], [], frameon=False)
plt.tight_layout()
plt.show()


Now we can also plot the centroids of each tracked cell, in a Cartopy map, and add circles around the centroid based on how big the cell volume is in each timestep. We will also show the different cells ('SimpleNum') in different colors.

In [ ]:
df0_min_dbz = df0_min_dbz.sort_values(['SimpleNum', 'date_utc'])

# Convert lat/lon columns to numeric, coercing errors to NaN
df0_min_dbz['VolCentroidLat(deg)'] = pd.to_numeric(df0_min_dbz['VolCentroidLat(deg)'], errors='coerce')
df0_min_dbz['VolCentroidLon(deg)'] = pd.to_numeric(df0_min_dbz['VolCentroidLon(deg)'], errors='coerce')

# Set up map plot
plt.figure(figsize=(14, 10))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines(resolution='10m')
ax.add_feature(cfeature.BORDERS, linestyle=':')
gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 14}
gl.ylabel_style = {'size': 14}

# Force SimpleNum to categorical (string) for consistent labeling
df0_min_dbz['SimpleNum'] = df0_min_dbz['SimpleNum'].astype(str)

# Unique SimpleNum and colors
simple_nums = df0_min_dbz['SimpleNum'].unique()
palette = sns.color_palette("gist_ncar", n_colors=len(simple_nums))
df0_min_dbz['Volume(km3)'] = pd.to_numeric(df0_min_dbz['Volume(km3)'], errors='coerce').fillna(0)

for i, simple_num in enumerate(simple_nums):
    track = df0_min_dbz[df0_min_dbz['SimpleNum'] == simple_num].copy()
    lat = track['VolCentroidLat(deg)']
    lon = track['VolCentroidLon(deg)']
    vol = pd.to_numeric(track['Volume(km3)'], errors='coerce').fillna(0)
    sizes = np.log10(vol + 1) * 200

    ax.plot(lon, lat, marker='o', linestyle='-', color=palette[i], alpha=0.7,
             label=simple_num, transform=ccrs.PlateCarree())
    ax.scatter(lon, lat, s=sizes, color=palette[i], alpha=0.5,
               transform=ccrs.PlateCarree(), edgecolor='k', linewidth=0.5)

    # Add time labels next to each centroid
    for lo, la, t in zip(lon, lat, track['date_utc']):
        if pd.notna(lo) and pd.notna(la) and pd.notna(t):
            ax.text(lo + 0.02, la + 0.02, t.strftime('%H:%M'),
                    fontsize=8, color=palette[i], transform=ccrs.PlateCarree(),
                    ha='left', va='bottom')

ax.legend(title='SimpleNum', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.title('Tracks for ComplexNum = '+complexnum_min_dbz, fontsize=16)
plt.tight_layout()
plt.show()


### **2.3.4 Examine radar data and storm polygons together**

Now let's plot the cartesian radar data at 16:13 UTC and overlap the polygons of the storms that we have identified with Titan.

In [ ]:
TIME_IDX = 0

FILEPATH = "./data/radar/cart/20170812/ncf_20170812_161332.nc"   # <-- set your file path here
FIELD_NAME = "DBZ"            

with nc.Dataset(FILEPATH, "r") as ds:
    x = ds.variables["lon0"][:]
    y = ds.variables["lat0"][:]

    var = ds.variables[FIELD_NAME]
    data_3d = var[TIME_IDX, :, :, :]  # all z-levels: shape (z0, y0, x0)

    fill_value = getattr(var, "_FillValue", None)
    if fill_value is not None:
        data_3d = np.ma.masked_equal(data_3d, fill_value)
    data_3d = np.ma.masked_invalid(data_3d)

    # Retrieve Max reflectivity in column ( max across the z-axis )
    data = data_3d.max(axis=0)  # shape (y0, x0)

    vmin = getattr(var, "min_value", -10.0)
    vmax = getattr(var, "max_value", 70.0)
    units = getattr(var, "units", FIELD_NAME)

    time_var = ds.variables["time"]
    raw_time = num2date(time_var[TIME_IDX], units=time_var.units,
                         calendar=getattr(time_var, "calendar", "standard"))
    plot_time = pd.Timestamp(raw_time.isoformat())



# Handle tz-naive/aware consistently
if plot_time.tz is None:
    plot_time = plot_time.tz_localize("UTC")
else:
    plot_time = plot_time.tz_convert("UTC")

plot_time_min = plot_time.floor("min")
print("Plotting field for time:", plot_time_min)
print(f"Max reflectivity in column — min/max over field: {data.min():.2f} / {data.max():.2f}")

# Correct dtypes for Dataframe
df_min_dbz['date_utc'] = pd.to_datetime(df_min_dbz['date_utc'], utc=True)
df_min_dbz['EnvelopeCentroidLon(deg)'] = pd.to_numeric(df_min_dbz['EnvelopeCentroidLon(deg)'], errors='coerce')
df_min_dbz['EnvelopeCentroidLat(deg)'] = pd.to_numeric(df_min_dbz['EnvelopeCentroidLat(deg)'], errors='coerce')

# Filter dataframe: match to the minute, within radar domain 
lon_min, lon_max = float(x.min()), float(x.max())
lat_min, lat_max = float(y.min()), float(y.max())

df_min_dbz_match = df_min_dbz[
    (df_min_dbz['date_utc'].dt.floor("min") == plot_time_min) &
    (df_min_dbz['EnvelopeCentroidLon(deg)'] >= lon_min) &
    (df_min_dbz['EnvelopeCentroidLon(deg)'] <= lon_max) &
    (df_min_dbz['EnvelopeCentroidLat(deg)'] >= lat_min) &
    (df_min_dbz['EnvelopeCentroidLat(deg)'] <= lat_max)
]

# plot
fig, ax = plt.subplots(figsize=(12, 12), subplot_kw={'projection': ccrs.PlateCarree()})

ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

mesh = ax.pcolormesh(
    x, y, data,
    transform=ccrs.PlateCarree(),
    shading="auto", cmap="turbo",
    vmin=vmin, vmax=vmax,
    zorder=1,
)
plt.colorbar(mesh, ax=ax, label=units, shrink=0.7)

ax.coastlines(zorder=3)

gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 14}
gl.ylabel_style = {'size': 14}

#  Overlay only the matching polygons 
palette = sns.color_palette("gist_ncar", n_colors=max(len(df_min_dbz_match), 1))

for i, (idx, row) in enumerate(df_min_dbz_match.iterrows()):
    lat_centroid = float(row['EnvelopeCentroidLat(deg)'])
    lon_centroid = float(row['EnvelopeCentroidLon(deg)'])
    rays = row['nPolySidesPolygonRays']

    if not rays or len(rays) == 0:
        continue

    angles = np.deg2rad(np.arange(0, 360, 5))
    rays = np.array(rays, dtype=float)

    ray_x = rays * np.sin(angles)
    ray_y = rays * np.cos(angles)
    lat_vertices = lat_centroid + ray_y / 111
    lon_vertices = lon_centroid + ray_x / (111 * np.cos(np.deg2rad(lat_centroid)))
    polygon_points = list(zip(lon_vertices, lat_vertices))

    poly = Polygon(polygon_points)
    ax.add_geometries([poly], crs=ccrs.PlateCarree(),
                       edgecolor='black', facecolor='none', linewidth=1.5,
                       zorder=4)

    ax.plot(lon_centroid, lat_centroid, marker='o', color='black', markersize=3,
             transform=ccrs.PlateCarree(), zorder=5)

plt.title(f"Max Reflectivity in Column with Track Polygons — {plot_time_min}", fontsize=16)
plt.tight_layout()
plt.show()

## **3. Run TITAN storm tracking - storm size**

In this section, we will focus on the *min_storm_size* parameter. We have provided all other basic parameters including variable names, directories, etc.

A suggested range of parameter values is: 5 to 100 km^3. But feel free to try out different values to see how the analysis is affected.

<div class="alert alert-block alert-info"> <b>File Task: modify min_storm_size parameter in text files.</b> 

To modify min_storm_size, open the parameter file: ./params/Titan.params.size, which you can edit directly in JupyterLab. Make sure to save the file.

</div>

Run TITAN on the same gridded data as in Titan_tutorial.ipynb to identify and track the storms.

NOTE: *TITAN requires a sounding to convert reflectivity data into meaningful storm metrics for the analysis. The sounding can be ingested in SPDB format or entered manually. In our case, we manually entered 10 levels retrieved from an ERA5 sounding corresponding to the radar's location and the date and time of this case. You can verify this by inspecting the parameter files and searching for "sounding_mode = SPECIFY_SOUNDING;", along with the manually entered array in that section.*

<div class="alert alert-block alert-warning"> <b>Task #1: Run Titan on the Jastrebac data with your chosen min_storm_size value.</b>
<br>
    Note the different parameter file name and output directory.
    <br>
<code lang="bash">!$LROSE_DIR/Titan -params ./params/Titan.params.size -start "2017 08 12 16 00 00" -end "2017 08 12 17 00 00" -odir ./data/titan_size/storms/ -debug</code>

</div>

In [ ]:
# run Titan
!$LROSE_DIR/Titan -params ./params/Titan.params.size -start "2017 08 12 16 00 00" -end "2017 08 12 17 00 00" -odir ./data/titan_size/storms/ -debug  


### **3.2. Convert TITAN binary output to readable format - storm size**

The TITAN output is in a binary format. In order to read the data, we first convert the TITAN output to an ASCII file.

<div class="alert alert-block alert-warning"> <b>Task #2: Convert Titan binary output to ASCII.</b>

<code lang="bash">!$LROSE_DIR/Tracks2Ascii -params ./params/Tracks2Ascii.params -f ./data/titan_size/storms/20170812.th5 > ./data/titan_size/ascii/Tracks2Ascii20170812.txt -debug</code>

</div>

In [ ]:
!$LROSE_DIR/Tracks2Ascii -params ./params/Tracks2Ascii.params -f ./data/titan_size/storms/20170812.th5 > ./data/titan_size/ascii/Tracks2Ascii20170812.txt -debug


### **3.3. Investigate Output - storm size**

We'll load the necessary Python packages and plot some of the TITAN output now.

In [ ]:
# Import Python packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from shapely.geometry import Polygon
from netCDF4 import num2date
import netCDF4 as nc

In [ ]:
# set the path for the ASCII file
file_size = "./data/titan_size/ascii/Tracks2Ascii20170812.txt"

### **3.3.1 Read the ASCII file into a pandas dataframe**

Open text file and adjust columns names in order to import to a pandas dataframe. 
Since the text file has irregular delimiters, we need to add some extra steps.

In [ ]:
##open file and extract column names
f_size = open(file_size)
lines_size = f_size.readlines()
f_size.close()

label_line_index = None  

for i, line in enumerate(lines_size):
    if 'labels' in line:
        label_line_index = i
        break  
labels_size = lines_size[label_line_index].split(":", 1)[1].strip().split(",")

In [ ]:
#the data lines are the ones that do not start with #
data_lines_size = [line.strip() for line in lines_size if not line.startswith("#")]

The file last three rows are labeled "parents", "children", "nPolySidesPolygonRays*72". 

Parents and children columns refer to identifiers based on storm merging and splitting processes. The Polygon column shows the values for the lines from the polygon centroid to each vertex, in km. There are 72 values because each line is separated 5 deg (72*5 =360). 

With that information and the "envelope_centroid" column, we can retrieve the cells envelopes at each timestep.

In [ ]:
rows_size = []
for line in data_lines_size:
    parts = line.split()

    try:
        # Try parsing the polygon count value (always right before 72 values)
        poly_count_index = -73  # 72 floats + 1 count (the column starts with the numnber 72, which is not part of the values)

        # Parents and children may be missing
        parent_str = parts[poly_count_index - 2]
        child_str = parts[poly_count_index - 1]

        # Handle missing values marked as "-"
        parents = int(parent_str) if parent_str != '-' else np.nan
        children = int(child_str) if child_str != '-' else np.nan

        # Polygon values: skip the count, get the next 72 values
        polygon_values = list(map(float, parts[poly_count_index + 1:]))

        # Fixed columns
        fixed_cols = parts[:poly_count_index - 2]

        # Combine into one row
        row = fixed_cols + [parents, children, polygon_values]
        rows_size.append(row)
    except Exception as e:
        continue

In [ ]:
# Final columns: fixed + 3 custom ones
final_labels_size = labels_size[:len(rows_size[0]) - 3] + ['parents', 'children', 'nPolySidesPolygonRays']

# Create DataFrame
df_size = pd.DataFrame(rows_size, columns=final_labels_size)

# Convert date and time columns to datetime
df_size['date_utc'] = pd.to_datetime(
    df_size['Year'].astype(str) + '-' + df_size['Month'].astype(str).str.zfill(2) + '-' +
    df_size['Day'].astype(str).str.zfill(2) + ' ' + df_size['Hour'].astype(str).str.zfill(2) + ':' +
    df_size['Min'].astype(str).str.zfill(2) + ':' + df_size['Sec'].astype(str).str.zfill(2),
    format='%Y-%m-%d %H:%M:%S', utc=True
)
# Print df_size 
print(df_size)

Let's explore the TITAN output now!

### **3.3.2 Examine the number of identified individual storms per complex**

TITAN outputs cell features at each tracking timestep and identifies individual cells within larger systems based on their interaction with neighboring cells. As a result, each cell in the TITAN output ASCII file is assigned two identifiers: a "SimpleNum" and a "ComplexNum". For example, when tracking a multicell system, each individual cell within it will have a distinct "SimpleNum," but all cells belonging to that same system will share a common "ComplexNum."

Let's inspect our case now! How many Complexes can we identify? Which one contains more tracks (e.g., single cell tracks, and split/merge processes)?

In [ ]:
# Count number of unique SimpleNum per ComplexNum
simple_counts_size = df_size.groupby('ComplexNum')['SimpleNum'].nunique().reset_index(name='NumSimple')

# Sort (optional, for better visuals)
#simple_counts = simple_counts.sort_values(by='NumSimple', ascending=False)

# Plot 1
plt.figure(figsize=(10, 8))
sns.barplot(y='ComplexNum', x='NumSimple', data=simple_counts_size, palette='Set3')
plt.title('Number of SimpleNum per ComplexNum', fontsize=16)
plt.xlabel('Count of Unique SimpleNum ID', fontsize=14)
plt.ylabel('ComplexNum ID', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.grid(axis='x')
plt.show()


### **3.3.3 Explore a single complex**

Based on the figure above, let's now explore one of the Complex tracks. Look for a ComplexNum that has a lot of tracked individual storms. **Replace the number in the complexnum variable in the cell below.**

In our case, we will plot the Maximum Reflectivity (```'MaxDBZ(dBZ)'```) for the entire life cycle of this complex system. Each individual storm is plotted in a different color (see the ```hue='SimpleNum'``` parameter in the plot).

You can play and choose another attribute (e.g., Echo Top, Vil) from the ASCII file, and see how the attributes vary. **Copy the variable name from the ASCII file and replace the string in the plotvar variable in the cell below.**

In [1]:
# MODIFY THESE VARIABLES AS NEEDED

complexnum_size = "17" # should be a string!
plotvar_size = 'MaxDBZ(dBZ)'

In [ ]:
#  Filter dataframe for the chosen ComplexNum and sort by time
df0_size = df_size[df_size['ComplexNum'] == complexnum_size].copy()
df0_size = df0_size.sort_values('date_utc')
df0_size[plotvar] = pd.to_numeric(df0_size[plotvar_size], errors='coerce')
df0_size['date_utc'] = pd.to_datetime(df0_size['date_utc'], errors='coerce', utc=True)

#  Plot
y_min = 30
y_max = 70
y_ticks = np.arange(y_min, y_max + 1, 5)

plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")
sns.lineplot(data=df0_size_size, x='date_utc', y=plotvar_size, hue='SimpleNum', palette='gist_ncar')
plt.ylim(y_min, y_max)
plt.yticks(y_ticks,fontsize=12)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.xticks(fontsize=12)
plt.title(plotvar_size+' over Time for ComplexNum = '+complexnum_size, fontsize=16)
plt.xlabel('Time (UTC)', fontsize=14)
plt.ylabel(plotvar_size,fontsize=14)
# Remove legend
plt.legend([], [], frameon=False)
plt.tight_layout()
plt.show()


Now we can also plot the centroids of each tracked cell, in a Cartopy map, and add circles around the centroid based on how big the cell volume is in each timestep. We will also show the different cells ('SimpleNum') in different colors.

In [ ]:
df0_size = df0_size.sort_values(['SimpleNum', 'date_utc'])

# Convert lat/lon columns to numeric, coercing errors to NaN
df0_size['VolCentroidLat(deg)'] = pd.to_numeric(df0_size['VolCentroidLat(deg)'], errors='coerce')
df0_size['VolCentroidLon(deg)'] = pd.to_numeric(df0_size['VolCentroidLon(deg)'], errors='coerce')

# Set up map plot
plt.figure(figsize=(14, 10))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines(resolution='10m')
ax.add_feature(cfeature.BORDERS, linestyle=':')
gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 14}
gl.ylabel_style = {'size': 14}

# Force SimpleNum to categorical (string) for consistent labeling
df0_size['SimpleNum'] = df0_size['SimpleNum'].astype(str)

# Unique SimpleNum and colors
simple_nums = df0_size['SimpleNum'].unique()
palette = sns.color_palette("gist_ncar", n_colors=len(simple_nums))
df0_size['Volume(km3)'] = pd.to_numeric(df0_size['Volume(km3)'], errors='coerce').fillna(0)

for i, simple_num in enumerate(simple_nums):
    track = df0_size[df0_size['SimpleNum'] == simple_num].copy()
    lat = track['VolCentroidLat(deg)']
    lon = track['VolCentroidLon(deg)']
    vol = pd.to_numeric(track['Volume(km3)'], errors='coerce').fillna(0)
    sizes = np.log10(vol + 1) * 200

    ax.plot(lon, lat, marker='o', linestyle='-', color=palette[i], alpha=0.7,
             label=simple_num, transform=ccrs.PlateCarree())
    ax.scatter(lon, lat, s=sizes, color=palette[i], alpha=0.5,
               transform=ccrs.PlateCarree(), edgecolor='k', linewidth=0.5)

    # Add time labels next to each centroid
    for lo, la, t in zip(lon, lat, track['date_utc']):
        if pd.notna(lo) and pd.notna(la) and pd.notna(t):
            ax.text(lo + 0.02, la + 0.02, t.strftime('%H:%M'),
                    fontsize=8, color=palette[i], transform=ccrs.PlateCarree(),
                    ha='left', va='bottom')

ax.legend(title='SimpleNum', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.title('Tracks for ComplexNum = '+complexnum, fontsize=16)
plt.tight_layout()
plt.show()


### **2.3.4 Examine radar data and storm polygons together**

Now let's plot the cartesian radar data at 16:13 UTC and overlap the polygons of the storms that we have identified with Titan.

In [ ]:
TIME_IDX = 0

FILEPATH = "./data/radar/cart/20170812/ncf_20170812_161332.nc"   # <-- set your file path here
FIELD_NAME = "DBZ"            

with nc.Dataset(FILEPATH, "r") as ds:
    x = ds.variables["lon0"][:]
    y = ds.variables["lat0"][:]

    var = ds.variables[FIELD_NAME]
    data_3d = var[TIME_IDX, :, :, :]  # all z-levels: shape (z0, y0, x0)

    fill_value = getattr(var, "_FillValue", None)
    if fill_value is not None:
        data_3d = np.ma.masked_equal(data_3d, fill_value)
    data_3d = np.ma.masked_invalid(data_3d)

    # Retrieve Max reflectivity in column ( max across the z-axis )
    data = data_3d.max(axis=0)  # shape (y0, x0)

    vmin = getattr(var, "min_value", -10.0)
    vmax = getattr(var, "max_value", 70.0)
    units = getattr(var, "units", FIELD_NAME)

    time_var = ds.variables["time"]
    raw_time = num2date(time_var[TIME_IDX], units=time_var.units,
                         calendar=getattr(time_var, "calendar", "standard"))
    plot_time = pd.Timestamp(raw_time.isoformat())



# Handle tz-naive/aware consistently
if plot_time.tz is None:
    plot_time = plot_time.tz_localize("UTC")
else:
    plot_time = plot_time.tz_convert("UTC")

plot_time_min = plot_time.floor("min")
print("Plotting field for time:", plot_time_min)
print(f"Max reflectivity in column — min/max over field: {data.min():.2f} / {data.max():.2f}")

# Correct dtypes for Dataframe
df_size['date_utc'] = pd.to_datetime(df_size['date_utc'], utc=True)
df_size['EnvelopeCentroidLon(deg)'] = pd.to_numeric(df_size['EnvelopeCentroidLon(deg)'], errors='coerce')
df_size['EnvelopeCentroidLat(deg)'] = pd.to_numeric(df_size['EnvelopeCentroidLat(deg)'], errors='coerce')

# Filter dataframe: match to the minute, within radar domain 
lon_min, lon_max = float(x.min()), float(x.max())
lat_min, lat_max = float(y.min()), float(y.max())

df_size_match = df_size[
    (df_size['date_utc'].dt.floor("min") == plot_time_min) &
    (df_size['EnvelopeCentroidLon(deg)'] >= lon_min) &
    (df_size['EnvelopeCentroidLon(deg)'] <= lon_max) &
    (df_size['EnvelopeCentroidLat(deg)'] >= lat_min) &
    (df_size['EnvelopeCentroidLat(deg)'] <= lat_max)
]

# plot
fig, ax = plt.subplots(figsize=(12, 12), subplot_kw={'projection': ccrs.PlateCarree()})

ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

mesh = ax.pcolormesh(
    x, y, data,
    transform=ccrs.PlateCarree(),
    shading="auto", cmap="turbo",
    vmin=vmin, vmax=vmax,
    zorder=1,
)
plt.colorbar(mesh, ax=ax, label=units, shrink=0.7)

ax.coastlines(zorder=3)

gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 14}
gl.ylabel_style = {'size': 14}

#  Overlay only the matching polygons 
palette = sns.color_palette("gist_ncar", n_colors=max(len(df_size_match), 1))

for i, (idx, row) in enumerate(df_size_match.iterrows()):
    lat_centroid = float(row['EnvelopeCentroidLat(deg)'])
    lon_centroid = float(row['EnvelopeCentroidLon(deg)'])
    rays = row['nPolySidesPolygonRays']

    if not rays or len(rays) == 0:
        continue

    angles = np.deg2rad(np.arange(0, 360, 5))
    rays = np.array(rays, dtype=float)

    ray_x = rays * np.sin(angles)
    ray_y = rays * np.cos(angles)
    lat_vertices = lat_centroid + ray_y / 111
    lon_vertices = lon_centroid + ray_x / (111 * np.cos(np.deg2rad(lat_centroid)))
    polygon_points = list(zip(lon_vertices, lat_vertices))

    poly = Polygon(polygon_points)
    ax.add_geometries([poly], crs=ccrs.PlateCarree(),
                       edgecolor='black', facecolor='none', linewidth=1.5,
                       zorder=4)

    ax.plot(lon_centroid, lat_centroid, marker='o', color='black', markersize=3,
             transform=ccrs.PlateCarree(), zorder=5)

plt.title(f"Max Reflectivity in Column with Track Polygons — {plot_time_min}", fontsize=16)
plt.tight_layout()
plt.show()